In [25]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go

In [26]:
#save and display plots from error angle and distance to bead

In [27]:
# Create folder for chase metric plots
fig_dir = r"./../../dataFolders/MuscaChasingBeads/Figures/ChaseMetrics/"
os.makedirs(fig_dir, exist_ok=True)

# Where CHASE_METRICS CSVs live
INPUT_DIR  = r"./../../dataFolders/MuscaChasingBeads/ErrorAngle_DistFromBead/"
SMOOTH_DIR = r"./../../dataFolders/MuscaChasingBeads/xyz_smooth/"
fig_dir    = r"./../../dataFolders/MuscaChasingBeads/Figures/ChaseMetrics"
os.makedirs(fig_dir, exist_ok=True)

# ==========================================================
# FIND CHASE METRICS FILES
# ==========================================================
csv_files = sorted(glob.glob(os.path.join(INPUT_DIR, "*_CHASE_METRICS.csv")))
if not csv_files:
    raise FileNotFoundError("No CHASE_METRICS CSV files found.")

In [29]:

# ==========================================================
# PLOTTING
# ==========================================================
for path in csv_files:
    base = os.path.splitext(os.path.basename(path))[0]
    print("Plotting:", base)
    trial_name = '_'.join(base.split("_")[0:2])[:-6]

    # --- Create trial-specific subfolder ---
    trial_save_dir = os.path.join(FIG_BASE_DIR, trial_name)
    os.makedirs(trial_save_dir, exist_ok=True)

    df = pd.read_csv(path)
    t  = df["frame"].values

    # ================= 3D RESULTANT =================
    fig, ax1 = plt.subplots(figsize=(14, 6))
    ax1.plot(t, df["dist_head_m"], color="darkred", linewidth=2)
    ax1.set_xlabel("Frame")
    ax1.set_ylabel("Distance (m)", color="darkred")
    ax1.tick_params(axis="y", labelcolor="darkred")
    ax1.grid(True)
    ax2 = ax1.twinx()
    ax2.plot(t, df["error_angle_head_deg"], color="purple", linewidth=2)
    ax2.set_ylabel("Error Angle (°)", color="purple")
    ax2.tick_params(axis="y", labelcolor="purple")
    plt.title(f"{trial_name} — 3D Resultant: Distance + Error Angle")
    plt.tight_layout()
    plt.savefig(os.path.join(trial_save_dir, f"{trial_name}_resultant.png"), dpi=300)
    plt.close()

    # ================= HORIZONTAL (XY) =================
    fig, ax = plt.subplots(figsize=(14, 6))
    ax.plot(t, df["error_angle_horiz_deg"], color="steelblue", linewidth=2)
    ax.set_xlabel("Frame")
    ax.set_ylabel("Horizontal Error Angle (°)", color="steelblue")
    ax.tick_params(axis="y", labelcolor="steelblue")
    ax.grid(True)
    plt.title(f"{trial_name} — Horizontal Error Angle (XY)")
    plt.tight_layout()
    plt.savefig(os.path.join(trial_save_dir, f"{trial_name}_horizontal.png"), dpi=300)
    plt.close()

    # ================= VERTICAL (YZ) =================
    fig, ax = plt.subplots(figsize=(14, 6))
    ax.plot(t, df["error_angle_vert_deg"], color="seagreen", linewidth=2)
    ax.set_xlabel("Frame")
    ax.set_ylabel("Vertical Error Angle (°)", color="seagreen")
    ax.tick_params(axis="y", labelcolor="seagreen")
    ax.grid(True)
    plt.title(f"{trial_name} — Vertical Error Angle (YZ)")
    plt.tight_layout()
    plt.savefig(os.path.join(trial_save_dir, f"{trial_name}_vertical.png"), dpi=300)
    plt.close()

    # ================= 3D INTERACTIVE =================
    smooth_path = glob.glob(os.path.join(SMOOTH_DIR, f"{trial_name}*_SMOOTH.csv"))
    if smooth_path:
        df_s = pd.read_csv(smooth_path[0])

        hx, hy, hz = df_s["pt2_X"].values, df_s["pt2_Y"].values, df_s["pt2_Z"].values
        bx, by, bz = df_s["pt1_X"].values, df_s["pt1_Y"].values, df_s["pt1_Z"].values
        err        = df["error_angle_head_deg"].values
        frames     = df["frame"].values

        # Fly trajectory coloured by error angle
        fly_trace = go.Scatter3d(
            x=hx, y=hy, z=hz,
            mode="lines+markers",
            name="Fly (Head)",
            marker=dict(
                size=3,
                color=err,
                colorscale="Turbo",
                colorbar=dict(title="Error Angle (°)"),
                showscale=True,
            ),
            line=dict(color=err, colorscale="Turbo", width=3),
            hovertemplate=(
                "Frame: %{text}<br>"
                "X: %{x:.4f}<br>Y: %{y:.4f}<br>Z: %{z:.4f}<br>"
                "Error Angle: %{marker.color:.2f}°"
                "<extra>Fly</extra>"
            ),
            text=frames,
        )

        # Bead trajectory
        bead_trace = go.Scatter3d(
            x=bx, y=by, z=bz,
            mode="lines",
            name="Bead",
            line=dict(color="grey", width=2, dash="dot"),
            hovertemplate=(
                "Frame: %{text}<br>"
                "X: %{x:.4f}<br>Y: %{y:.4f}<br>Z: %{z:.4f}"
                "<extra>Bead</extra>"
            ),
            text=frames,
        )

        # Connecting lines subsampled every 10 frames
        step = 10
        line_x, line_y, line_z = [], [], []
        for i in range(0, len(hx), step):
            line_x += [hx[i], bx[i], None]
            line_y += [hy[i], by[i], None]
            line_z += [hz[i], bz[i], None]

        connect_trace = go.Scatter3d(
            x=line_x, y=line_y, z=line_z,
            mode="lines",
            name="Head → Bead",
            line=dict(color="rgba(180,180,180,0.4)", width=1),
            hoverinfo="skip",
        )

        fig_3d = go.Figure(data=[bead_trace, connect_trace, fly_trace])
        fig_3d.update_layout(
            title=f"{trial_name} — 3D Chase (coloured by Error Angle)",
            scene=dict(
                xaxis_title="X",
                yaxis_title="Y",
                zaxis_title="Z",
            ),
            legend=dict(x=0, y=1),
            width=1100,
            height=750,
        )

        out_html = os.path.join(trial_save_dir, f"{trial_name}_3D_error.html")
        fig_3d.write_html(out_html)
        print(f"  Saved 3D plot: {out_html}")
    else:
        print(f"  WARNING: No SMOOTH file found for {trial_name} — skipping 3D plot")

    print(f"  All plots saved to: {trial_save_dir}")

print("\nChase metrics plots saved trial-wise.")

Plotting: Trial2_180rpmxyzpts_CHASE_METRICS
  Saved 3D plot: ./../../dataFolders/MuscaChasingBeads/Figures/ChaseMetrics/Trial2_180rpm\Trial2_180rpm_3D_error.html
  All plots saved to: ./../../dataFolders/MuscaChasingBeads/Figures/ChaseMetrics/Trial2_180rpm
Plotting: Trial2_200rpmxyzpts_CHASE_METRICS
  Saved 3D plot: ./../../dataFolders/MuscaChasingBeads/Figures/ChaseMetrics/Trial2_200rpm\Trial2_200rpm_3D_error.html
  All plots saved to: ./../../dataFolders/MuscaChasingBeads/Figures/ChaseMetrics/Trial2_200rpm
Plotting: Trial3_180rpmxyzpts_CHASE_METRICS
  Saved 3D plot: ./../../dataFolders/MuscaChasingBeads/Figures/ChaseMetrics/Trial3_180rpm\Trial3_180rpm_3D_error.html
  All plots saved to: ./../../dataFolders/MuscaChasingBeads/Figures/ChaseMetrics/Trial3_180rpm
Plotting: Trial4_400rpmxyzpts_CHASE_METRICS
  Saved 3D plot: ./../../dataFolders/MuscaChasingBeads/Figures/ChaseMetrics/Trial4_400rpm\Trial4_400rpm_3D_error.html
  All plots saved to: ./../../dataFolders/MuscaChasingBeads/Figures